# Fase 3: Carga al Data Warehouse (Load)
En esta etapa, tomaremos los conjuntos de datos limpios y los estructuraremos bajo un **Modelo Estrella (Star Schema)**.
Estructura del Data Warehouse local:
- **Dimensiones (Dim)**: Contienen el contexto del negocio (Tiempo, Clientes, Productos, Sucursales, Canales de venta, Bodegas, Métodos de pago, Campañas).
- **Tablas de Hechos (Fact)**: Contienen las métricas cuantitativas del negocio asociadas con claves foráneas (llaves subrogadas) a las dimensiones (Ventas, Inventario, Movimientos de Inventario, Marketing).

Posteriormente, cargaremos este modelo a una base de datos local en **SQLite** y finalmente sincronizaremos los datos con **Google BigQuery**.

In [ ]:
import os
import json
import sqlite3
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv


## 1. Carga y Preparación Inicial de Datos Limpios
Aplicamos de manera integrada las transformaciones previas para generar los conjuntos limpios listos para modelar.

In [ ]:
DATA_DIR = "../data"
DW_PATH = os.path.join(DATA_DIR, "datacommerce_dw.db")

# Clientes
df_clientes = pd.read_json(os.path.join(DATA_DIR, "clientes.json"))
df_cli_limpio = df_clientes.drop_duplicates(subset=["cliente_id"], keep="last").copy()
df_cli_limpio["municipio"] = df_cli_limpio["municipio"].replace("", "No disponible").fillna("No disponible")
df_cli_limpio["fecha_registro"] = pd.to_datetime(df_cli_limpio["fecha_registro"], errors="coerce").dt.strftime("%Y-%m-%d")
for col in df_cli_limpio.select_dtypes(include="object").columns:
    df_cli_limpio[col] = df_cli_limpio[col].fillna("No disponible")
df_cli_limpio["edad"] = pd.to_numeric(df_cli_limpio["edad"], errors="coerce").fillna(0).astype(int)

# Productos
df_productos = pd.read_excel(os.path.join(DATA_DIR, "productos.xlsx"))
df_prod_limpio = df_productos.drop_duplicates(subset=["producto_id"], keep="last").copy()
df_prod_limpio["costo_unitario"] = pd.to_numeric(df_prod_limpio["costo_unitario"], errors="coerce").fillna(0.0)
df_prod_limpio["precio_lista"] = pd.to_numeric(df_prod_limpio["precio_lista"], errors="coerce").fillna(0.0)
for col in df_prod_limpio.select_dtypes(include="object").columns:
    df_prod_limpio[col] = df_prod_limpio[col].astype(str).str.strip().replace("", "No disponible").fillna("No disponible")

# Ventas
df_ventas = pd.read_csv(os.path.join(DATA_DIR, "ventas.csv"))
def _parsear_fecha(valor):
    texto = str(valor).strip()
    if not texto or texto.lower() == "nan": return pd.NaT
    if "-" in texto and len(texto.split("-")[0]) == 4: return pd.to_datetime(texto, format="%Y-%m-%d")
    if "/" in texto:
        partes = texto.split("/")
        if len(partes[0]) == 4: return pd.to_datetime(texto, format="%Y/%m/%d")
        return pd.to_datetime(texto, format="%d/%m/%Y")
    return pd.to_datetime(texto, errors="coerce")

df_ventas_limpio = df_ventas.drop_duplicates(subset=["venta_id"], keep="last").copy()
df_ventas_limpio["fecha_venta"] = df_ventas_limpio["fecha_venta"].apply(_parsear_fecha).dt.strftime("%Y-%m-%d")
df_ventas_limpio["canal_venta"] = df_ventas_limpio["canal_venta"].astype(str).str.strip().str.title().replace({"Web": "E-commerce"})
df_ventas_limpio["metodo_pago"] = df_ventas_limpio["metodo_pago"].fillna("No disponible")
for col in ["cantidad", "precio_unitario", "descuento", "total_venta"]:
    df_ventas_limpio[col] = pd.to_numeric(df_ventas_limpio[col], errors="coerce").fillna(0)
df_ventas_limpio = df_ventas_limpio.dropna(subset=["venta_id", "cliente_id", "producto_id", "fecha_venta"])

# Inventario
with sqlite3.connect(os.path.join(DATA_DIR, "inventario.db")) as conn:
    df_inventario_actual = pd.read_sql_query("SELECT * FROM inventario_actual;", conn)
    df_movimientos = pd.read_sql_query("SELECT * FROM movimientos_inventario;", conn)
df_inv_limpio = df_inventario_actual.drop_duplicates(subset=["producto_id", "bodega"], keep="last").copy()
df_inv_limpio["bodega"] = df_inv_limpio["bodega"].astype(str).str.strip().str.title()
df_inv_limpio["existencia"] = pd.to_numeric(df_inv_limpio["existencia"], errors="coerce").fillna(0).astype(int)
df_mov_limpio = df_movimientos.drop_duplicates(subset=["id"], keep="last").copy()
df_mov_limpio["fecha"] = pd.to_datetime(df_mov_limpio["fecha"], errors="coerce").dt.strftime("%Y-%m-%d")
df_mov_limpio["tipo"] = df_mov_limpio["tipo"].astype(str).str.strip().str.title()
df_mov_limpio["cantidad"] = pd.to_numeric(df_mov_limpio["cantidad"], errors="coerce").fillna(0).astype(int)
df_mov_limpio = df_mov_limpio.dropna(subset=["fecha", "producto_id"])

# Marketing
with open(os.path.join(DATA_DIR, "api_marketing_response.json"), encoding="utf-8") as f:
    df_marketing = pd.json_normalize(json.load(f), record_path="campaigns")
df_mkt_limpio = df_marketing.copy()
if "campaña_id" in df_mkt_limpio.columns:
    df_mkt_limpio = df_mkt_limpio.rename(columns={"campaña_id": "campana_id"})
if "campa\u00f1a_id" in df_mkt_limpio.columns:
    df_mkt_limpio = df_mkt_limpio.rename(columns={"campa\u00f1a_id": "campana_id"})
df_mkt_limpio = df_mkt_limpio.drop_duplicates(subset=["campana_id"], keep="last").copy()
df_mkt_limpio["fecha"] = pd.to_datetime(df_mkt_limpio["fecha"], errors="coerce").dt.strftime("%Y-%m-%d")
df_mkt_limpio["plataforma"] = df_mkt_limpio["plataforma"].astype(str).str.strip().str.title()
for col in ["impresiones", "clics", "leads", "conversiones"]:
    df_mkt_limpio[col] = pd.to_numeric(df_mkt_limpio[col], errors="coerce").fillna(0).astype(int)
df_mkt_limpio["costo"] = pd.to_numeric(df_mkt_limpio["costo"], errors="coerce").fillna(0.0)

print("✅ Todos los datos listos para el modelamiento dimensional.")


## 2. Construcción de Dimensiones (Modelo Estrella)
Estructuramos las tablas que actuarán como dimensiones.

In [ ]:
# 2.1 Dimensión de Tiempo
def _fechas_unicas(*dataframes):
    fechas = set()
    for df, col in dataframes:
        if df is not None and col in df.columns:
            fechas.update(df[col].dropna().astype(str).tolist())
    return sorted(fechas)

fechas = _fechas_unicas((df_ventas_limpio, "fecha_venta"), (df_mov_limpio, "fecha"), (df_mkt_limpio, "fecha"))
registros_tiempo = []
for f in fechas:
    dt = pd.to_datetime(f)
    registros_tiempo.append({
        "fecha_key": int(dt.strftime("%Y%m%d")),
        "fecha": f,
        "anio": dt.year,
        "mes": dt.month,
        "dia": dt.day,
        "trimestre": (dt.month - 1) // 3 + 1,
        "nombre_mes": dt.strftime("%B"),
        "dia_semana": dt.strftime("%A"),
    })
dim_tiempo = pd.DataFrame(registros_tiempo)

# 2.2 Dimensión Cliente
dim_cliente = df_cli_limpio.copy().rename(columns={"cliente_id": "cliente_id_natural"})
clientes_ventas = set(df_ventas_limpio["cliente_id"].astype(int).unique())
clientes_dim = set(dim_cliente["cliente_id_natural"].astype(int).unique())
faltantes = clientes_ventas - clientes_dim
for c_id in sorted(faltantes):
    dim_cliente = pd.concat([dim_cliente, pd.DataFrame([{
        "cliente_id_natural": c_id, "nombre": "Cliente no registrado", "genero": "No disponible",
        "edad": 0, "departamento": "No disponible", "municipio": "No disponible", "fecha_registro": None,
        "segmento_cliente": "No disponible"
    }])], ignore_index=True)
dim_cliente.insert(0, "cliente_key", range(1, len(dim_cliente) + 1))

# 2.3 Dimensión Producto
dim_producto = df_prod_limpio.copy().rename(columns={"producto_id": "producto_id_natural"})
dim_producto.insert(0, "producto_key", range(1, len(dim_producto) + 1))

# 2.4 Dimensión Sucursal
SUCURSALES = {1: "Sucursal Centro", 2: "Sucursal Norte", 3: "Sucursal Sur"}
sucursales_ids = sorted(df_ventas_limpio["sucursal_id"].dropna().unique())
dim_sucursal = pd.DataFrame([{
    "sucursal_key": idx, "sucursal_id_natural": int(s_id), "nombre_sucursal": SUCURSALES.get(int(s_id), f"Sucursal {int(s_id)}")
} for idx, s_id in enumerate(sucursales_ids, start=1)])

# 2.5 Dimensión Canal
canales = sorted(df_ventas_limpio["canal_venta"].dropna().unique())
dim_canal = pd.DataFrame([{"canal_key": idx, "canal_venta": c} for idx, c in enumerate(canales, start=1)])

# 2.6 Dimensión Bodega
bodegas = sorted(df_inv_limpio["bodega"].dropna().unique())
dim_bodega = pd.DataFrame([{"bodega_key": idx, "nombre_bodega": b} for idx, b in enumerate(bodegas, start=1)])

# 2.7 Dimensión Método Pago
metodos = sorted(df_ventas_limpio["metodo_pago"].dropna().unique())
dim_metodo_pago = pd.DataFrame([{"metodo_pago_key": idx, "metodo_pago": m} for idx, m in enumerate(metodos, start=1)])

# 2.8 Dimensión Campaña Marketing
dim_campana = df_mkt_limpio[["campana_id", "plataforma"]].copy().rename(columns={"campana_id": "campana_id_natural"})
dim_campana.insert(0, "campana_key", range(1, len(dim_campana) + 1))

print(f"Dimensiones construidas: DimTiempo ({len(dim_tiempo)}), DimCliente ({len(dim_cliente)}), DimProducto ({len(dim_producto)})")


## 3. Construcción de Tablas de Hechos (Fact Tables)
Generamos las tablas de hechos enlazando mediante llaves subrogadas.

In [ ]:
def _lookup(df, natural_col, key_col, value):
    match = df.loc[df[natural_col] == value, key_col]
    return int(match.iloc[0]) if not match.empty else None

# 3.1 Fact Ventas
reg_ventas = []
for _, row in df_ventas_limpio.iterrows():
    cliente_key = _lookup(dim_cliente, "cliente_id_natural", "cliente_key", int(row["cliente_id"]))
    if cliente_key is None: continue
    reg_ventas.append({
        "venta_id": int(row["venta_id"]),
        "fecha_key": _lookup(dim_tiempo, "fecha", "fecha_key", row["fecha_venta"]),
        "cliente_key": cliente_key,
        "producto_key": _lookup(dim_producto, "producto_id_natural", "producto_key", int(row["producto_id"])),
        "sucursal_key": _lookup(dim_sucursal, "sucursal_id_natural", "sucursal_key", int(row["sucursal_id"])),
        "canal_key": _lookup(dim_canal, "canal_venta", "canal_key", row["canal_venta"]),
        "metodo_pago_key": _lookup(dim_metodo_pago, "metodo_pago", "metodo_pago_key", row["metodo_pago"]),
        "cantidad": int(row["cantidad"]),
        "precio_unitario": float(row["precio_unitario"]),
        "descuento": float(row["descuento"]),
        "total_venta": float(row["total_venta"]),
    })
fact_ventas = pd.DataFrame(reg_ventas)

# 3.2 Fact Inventario
reg_inv = []
for _, row in df_inv_limpio.iterrows():
    reg_inv.append({
        "producto_key": _lookup(dim_producto, "producto_id_natural", "producto_key", int(row["producto_id"])),
        "bodega_key": _lookup(dim_bodega, "nombre_bodega", "bodega_key", row["bodega"]),
        "existencia": int(row["existencia"]),
    })
fact_inventario = pd.DataFrame(reg_inv)

# 3.3 Fact Movimientos
reg_mov = []
for _, row in df_mov_limpio.iterrows():
    reg_mov.append({
        "movimiento_id": int(row["id"]),
        "fecha_key": _lookup(dim_tiempo, "fecha", "fecha_key", row["fecha"]),
        "producto_key": _lookup(dim_producto, "producto_id_natural", "producto_key", int(row["producto_id"])),
        "tipo_movimiento": row["tipo"],
        "cantidad": int(row["cantidad"]),
    })
fact_movimientos_inventario = pd.DataFrame(reg_mov)

# 3.4 Fact Marketing
reg_mkt = []
for _, row in df_mkt_limpio.iterrows():
    reg_mkt.append({
        "campana_key": _lookup(dim_campana, "campana_id_natural", "campana_key", int(row["campana_id"])),
        "fecha_key": _lookup(dim_tiempo, "fecha", "fecha_key", row["fecha"]),
        "impresiones": int(row["impresiones"]),
        "clics": int(row["clics"]),
        "costo": float(row["costo"]),
        "leads": int(row["leads"]),
        "conversiones": int(row["conversiones"]),
    })
fact_marketing = pd.DataFrame(reg_mkt)

print(f"Tablas de hechos construidas: FactVentas ({len(fact_ventas)}), FactInventario ({len(fact_inventario)}), FactMkt ({len(fact_marketing)})")


## 4. Carga al Data Warehouse Local (SQLite)
Insertamos las dimensiones y tablas de hechos en la base de datos local SQLite.

In [ ]:
print(f"Cargando tablas a SQLite en: {DW_PATH}")
dimensiones = {
    "dim_tiempo": dim_tiempo, "dim_cliente": dim_cliente, "dim_producto": dim_producto,
    "dim_sucursal": dim_sucursal, "dim_canal": dim_canal, "dim_bodega": dim_bodega,
    "dim_metodo_pago": dim_metodo_pago, "dim_campana": dim_campana
}
hechos = {
    "fact_ventas": fact_ventas, "fact_inventario": fact_inventario,
    "fact_movimientos_inventario": fact_movimientos_inventario, "fact_marketing": fact_marketing
}

with sqlite3.connect(DW_PATH) as conn:
    for nombre, df in dimensiones.items():
        df.to_sql(nombre, conn, if_exists="replace", index=False)
    for nombre, df in hechos.items():
        df.to_sql(nombre, conn, if_exists="replace", index=False)

print("✅ ¡Data Warehouse SQLite local actualizado exitosamente!")


## 5. Carga al Data Warehouse Cloud (Google BigQuery) - Opcional
Si dispones de un entorno en Google Cloud y has configurado tu variable de entorno `ID_PROYECTO`, ejecuta esta celda para migrar el modelo a BigQuery.

In [ ]:
load_dotenv()
ID_PROYECTO = os.getenv("ID_PROYECTO")
DATASET_ID = "proyectofinalG2"

if not ID_PROYECTO:
    print("⚠️ Variable 'ID_PROYECTO' no definida. No se ejecutará la carga a BigQuery.")
    print("Los datos permanecen almacenados en la base de datos local SQLite de forma exitosa.")
else:
    try:
        print(f"☁️ Conectando a GCP BigQuery con Proyecto: {ID_PROYECTO} y Dataset: {DATASET_ID}")
        client = bigquery.Client(project=ID_PROYECTO)
        
        tablas_dw = list(dimensiones.keys()) + list(hechos.keys())
        for tabla in tablas_dw:
            df_tabla = dimensiones[tabla] if tabla in dimensiones else hechos[tabla]
            tabla_destino = f"{ID_PROYECTO}.{DATASET_ID}.{tabla}"
            
            job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
            print(f" Subiendo {tabla} ({len(df_tabla)} filas)...")
            job = client.load_table_from_dataframe(df_tabla, tabla_destino, job_config=job_config)
            job.result()
            print(f"  ✅ Tabla {tabla} migrada exitosamente.")
        print("🏁 ¡Proceso de sincronización finalizado exitosamente!")
    except Exception as e:
        print(f"❌ Error durante la migración a BigQuery: {e}")
